# CT 타일 학습 — v4.1 nobig + **동분포(known-type) 이미지단위 split**

배포=알려진 47셀 타입 재검사 시나리오 → 셀 그룹분할이 아니라 **이미지 단위 stratified 분할**(모든 셀이 train·val 양쪽).
- baseline(plain seg head)과 **config·타일·레시피 전부 동일**, **split 방식만** 다름.
- eval = val 이미지(모델이 본 타입의 새 이미지)에 SAHI conf 스윕 → **동분포 F1**(남들 0.9x와 같은 시험지).
- held-out(0.73/0.86)을 대체하는 게 아니라 **다른 배포 시나리오의 숫자를 추가**. 둘 다 정직하게 병기.


In [1]:
# == §0 셋업 + CT 데이터 로케이트 (Colab v4.1) ==
!pip -q install ultralytics shapely pandas pyyaml
import os, shutil, subprocess, random, zipfile
from pathlib import Path
import numpy as np, pandas as pd
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

# 환경 바뀌면 아래 ROOT/DATA만 수정
ROOT      = Path('/content/drive/MyDrive/battery_yolo')
PROJECT   = ROOT/'빅프로젝트'                                         # 읽기전용(가중치 읽기용)
DATA      = ROOT/'data/battery_v4_1_output'
if not DATA.exists():
    cands=sorted((ROOT/'data').glob('*v4*1*')); assert cands, f'{DATA} 없음 — DATA 직접지정'
    DATA=cands[0]; print('자동탐색 DATA:', DATA)
DRIVE_OUT = ROOT/'kt_out_1'; DRIVE_OUT.mkdir(parents=True, exist_ok=True)
RUNS      = DRIVE_OUT/'runs_main'; RUNS.mkdir(parents=True, exist_ok=True)   # 학습출력(세션 죽어도 resume)

# manifest: loose 파일 우선, 없으면 zip 안에서 추출
def find_meta(name):
    hits=sorted(DATA.rglob(name))
    if hits: return hits[0]
    for z in sorted(DATA.rglob('*.zip')):
        try: zf=zipfile.ZipFile(z)
        except zipfile.BadZipFile: continue
        for n in zf.namelist():
            if n.replace('\\','/').split('/')[-1]==name:
                zf.extract(n,'/content/work/v41_meta'); return Path('/content/work/v41_meta')/n
    return None
MANIFEST=find_meta('manifest.csv'); assert MANIFEST, f'manifest 못찾음: {DATA}'
print('manifest:', MANIFEST)

# 동분포 split: 셀 그룹분할이 아니라 이미지 단위 → 47셀 모두 train·val 양쪽
mani = pd.read_csv(MANIFEST, dtype=str, keep_default_na=False)
seg  = mani[(mani['modality']=='CT') & (mani['included_seg'].str.lower()=='true')].copy()
dev  = seg                       # 알려진 타입 전량을 학습풀로
VAL_FRAC = 0.15
val_idx=[]
for cid,grp in dev.groupby('battery_id'):
    k = max(1,int(round(len(grp)*VAL_FRAC))) if len(grp)>=2 else 0   # 셀당 ≥1장은 반드시 train에 남김
    if k: val_idx += list(grp.sample(n=k, random_state=42).index)
vmask = dev.index.isin(val_idx)
ct_split = {'val': dev[vmask], 'train': dev[~vmask]}
print('CT 동분포 split | 전체', len(dev), '| train', len(ct_split['train']), '| val', len(ct_split['val']), '| 셀', dev['battery_id'].nunique())

# 파일명으로 인덱싱(name→이미지, stem→labels_seg) = zip 내부구조 무관.
# dev 커버리지 부족하면 CT zip을 채워질 때까지 순차 해제.
LOCAL = Path('/content/work/data/ct'); LOCAL.mkdir(parents=True, exist_ok=True)
IMG_EXTS={'.jpg','.jpeg','.png','.bmp','.tif','.tiff'}
def build_index():
    imgs={}; segs={}
    for f in LOCAL.rglob('*'):
        if not f.is_file(): continue
        sfx=f.suffix.lower()
        if sfx in IMG_EXTS: imgs[f.name]=f
        elif sfx=='.txt' and f.parent.name=='labels_seg': segs[f.stem]=f
    return imgs,segs
IMG_INDEX,SEG_INDEX=build_index()
def dev_cov(): return sum(1 for n in dev['output_image_name'] if n in IMG_INDEX)
if dev_cov() < len(dev)*0.95:
    zips=sorted(DATA.rglob('*CT*.zip')) or sorted(DATA.rglob('*.zip')); assert zips, f'CT zip 없음: {DATA}'
    for z in zips:
        print('해제:', z.name, '(dev 이미지 채울 때까지)')
        r=subprocess.run(['unzip','-q','-o',str(z),'-d',str(LOCAL)],capture_output=True,text=True)
        assert r.returncode<=1,(r.stderr or r.stdout)[-500:]
        IMG_INDEX,SEG_INDEX=build_index()
        if dev_cov()>=len(dev)*0.95: break
cov=dev_cov()
print('이미지 인덱스:',len(IMG_INDEX),'| labels_seg 인덱스:',len(SEG_INDEX),'| dev 커버리지:',f'{cov}/{len(dev)}')
assert cov>=len(dev)*0.95, f'dev 이미지 매칭 부족 {cov}/{len(dev)} — zip/구조 확인'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 73.0 MB/s eta 0:00:00
Mounted at /content/drive
자동탐색 DATA: /content/drive/MyDrive/battery_yolo/data/battery_v41_output
manifest: /content/drive/MyDrive/battery_yolo/data/battery_v41_output/reports/manifest.csv
CT 동분포 split | 전체 67605 | train 57465 | val 10140 | 셀 47
해제: battery_CT_v4_1_test.zip (dev 이미지 채울 때까지)
해제: battery_CT_v4_1_trainval.zip (dev 이미지 채울 때까지)
이미지 인덱스: 67607 | labels_seg 인덱스: 67605 | dev 커버리지: 67605/67605


In [2]:
# == §0.5 데이터 점검 (동분포/known-type 게이트) ==
tr_ids=set(ct_split['train']['battery_id']); va_ids=set(ct_split['val']['battery_id'])
print('CT 셀 =', len(tr_ids|va_ids), '(기대 47) | train셀', len(tr_ids), '| val셀', len(va_ids))
print('val 셀이 train에도 다 있나(동분포 핵심):', va_ids<=tr_ids, '| 양쪽 공통', len(tr_ids&va_ids))
assert (tr_ids|va_ids) and va_ids<=tr_ids, '동분포 게이트 실패: val 셀이 train에 없음(=held-out 되어버림)'
n_big=int((pd.to_numeric(seg['porosity_bbox_max_ratio'],errors='coerce').fillna(0)>=0.25).sum())
print('included_seg 대형(≥25%) 라벨:', n_big, '장 (기대 0 = v4.1 nobig)')
assert n_big==0, f'대형 오라벨 {n_big}장 잔존'
print('→ 동분포(known-type): 모든 셀 train·val 양쪽. eval=val 이미지(모델이 본 타입의 새 이미지) → 배포 대표숫자.')


CT 셀 = 47 (기대 47) | train셀 47 | val셀 47
val 셀이 train에도 다 있나(동분포 핵심): True | 양쪽 공통 47
included_seg 대형(≥25%) 라벨: 0 장 (기대 0 = v4.1 nobig)
→ 동분포(known-type): 모든 셀 train·val 양쪽. eval=val 이미지(모델이 본 타입의 새 이미지) → 배포 대표숫자.


In [2]:
# == §1 config (동분포: baseline과 동일 레시피, split만 다름) ==
SEED=42; random.seed(SEED)
CFG=dict(amp=True, lr0=0.0005, epochs=18, patience=15,
         hsv_h=0.0, hsv_s=0.0, hsv_v=0.1, degrees=10, flipud=0.5, translate=0.2,
         scale=0.1, copy_paste=0.0, mask_ratio=2, overlap_mask=False)
SAVE_PERIOD=1
DROP_BIG_LABELS=True; BIG_AREA_THR=0.25
RUN_NAME='train_ct_tiled_v41_samedist'
TILED=Path('/content/work/datasets/ct_tiled_v41_samedist_bg08')    # split 다르면 타일 새로 빌드(다른 split 타일과 섞이면 안 됨)
BACKUP=DRIVE_OUT/'ct_tiled_v41_samedist_best_backup.pt'
print(f'동분포 | DROP_BIG_LABELS={DROP_BIG_LABELS} | RUN_NAME={RUN_NAME}')
print('CFG:',CFG); print('tiled:',TILED,'| backup:',BACKUP)


동분포 | DROP_BIG_LABELS=True | RUN_NAME=train_ct_tiled_v41_samedist
CFG: {'amp': True, 'lr0': 0.0005, 'epochs': 18, 'patience': 15, 'hsv_h': 0.0, 'hsv_s': 0.0, 'hsv_v': 0.1, 'degrees': 10, 'flipud': 0.5, 'translate': 0.2, 'scale': 0.1, 'copy_paste': 0.0, 'mask_ratio': 2, 'overlap_mask': False}
tiled: /content/work/datasets/ct_tiled_v41_samedist_bg08 | backup: /content/drive/MyDrive/battery_yolo/kt_out_1/ct_tiled_v41_samedist_best_backup.pt


In [ ]:
# == §2 타일 데이터셋 빌드 (폭 네이티브 보존 + shapely 클립) ==
from PIL import Image
from shapely.geometry import Polygon, box as shbox
Image.MAX_IMAGE_PIXELS=None

def bbox_area(pts):
    xs=[p[0] for p in pts]; ys=[p[1] for p in pts]
    return (max(xs)-min(xs))*(max(ys)-min(ys))
def read_poly(lp):
    out=[]
    if lp is not None and lp.exists():
        for ln in lp.read_text().splitlines():
            v=ln.split()
            if len(v)>=7:
                cls=int(float(v[0])); xy=list(map(float,v[1:]))
                out.append((cls,list(zip(xy[0::2],xy[1::2]))))
    return out
def tile_starts(L,T,step):
    if L<=T: return [0]
    xs=list(range(0,L-T+1,step))
    if xs[-1]!=L-T: xs.append(L-T)
    return xs

Ws = pd.to_numeric(dev['roi_w'],errors='coerce').dropna()
Wmax=int(Ws.max())
print(f'dev {len(dev)}장 | roi 폭 min{int(Ws.min())} p50{int(Ws.median())} p95{int(Ws.quantile(.95))} max{Wmax}')
OVERLAP,BG_KEEP,N_BG_TILES = 0.2,0.08,1   # 배경 타일 비율. 올리면 precision↑ 학습시간↑
TILE=int(np.ceil(max(1280,Wmax)/32)*32)
assert TILE<=2048, f'폭 max {Wmax} → TILE {TILE} 과대. 이상치 확인'
print(f'-> TILE={TILE} (가로 1칸/세로 스트립, 폭 네이티브) | DROP_BIG_LABELS={DROP_BIG_LABELS}')

def clip_to_tile(polys,x0,y0,tw,th,W,H):
    tb=shbox(x0,y0,x0+tw,y0+th); out=[]
    for cls,pts in polys:
        ap=[(px*W,py*H) for px,py in pts]
        if len(ap)<3: continue
        g=Polygon(ap)
        if not g.is_valid: g=g.buffer(0)
        if g.is_empty: continue
        it=g.intersection(tb)
        if it.is_empty: continue
        for gg in (it.geoms if it.geom_type.startswith('Multi') else [it]):
            if gg.geom_type!='Polygon' or gg.area<4: continue
            ex=list(gg.exterior.coords)[:-1]
            out.append((cls,[(min(max((x-x0)/tw,0),1),min(max((y-y0)/th,0),1)) for x,y in ex]))
    return out
def slice_one(img_path,polys,split,tag):
    im=Image.open(img_path).convert('RGB'); W,H=im.size
    step=int(TILE*(1-OVERLAP))
    xs=tile_starts(W,TILE,step); ys=tile_starts(H,TILE,step)
    defect,empty=[],[]
    for y0 in ys:
        for x0 in xs:
            tw=min(TILE,W-x0); th=min(TILE,H-y0)
            kp=clip_to_tile(polys,x0,y0,tw,th,W,H)
            (defect if kp else empty).append((x0,y0,tw,th,kp))
    chosen=defect+[t for t in empty if random.random()<BG_KEEP] if defect else (random.sample(empty,min(N_BG_TILES,len(empty))) if empty else [])
    for x0,y0,tw,th,kp in chosen:
        fn=f'{tag}_{x0}_{y0}'
        im.crop((x0,y0,x0+tw,y0+th)).save(TILED/'images'/split/f'{fn}.jpg',quality=95)
        with open(TILED/'labels'/split/f'{fn}.txt','w') as f:
            for cls,npts in kp:
                f.write(str(cls)+' '+' '.join(f'{x:.6f} {y:.6f}' for x,y in npts)+'\n')
    return len(chosen),len(defect)

done=TILED/'.done'
if done.exists():
    print('스킵(이미 빌드됨):',TILED,'- 재빌드하려면 .done 삭제')
else:
    for split,rows in ct_split.items():
        (TILED/'images'/split).mkdir(parents=True,exist_ok=True)
        (TILED/'labels'/split).mkdir(parents=True,exist_ok=True)
        tot=dtot=miss=excl=0
        for img,stem in zip(rows['output_image_name'],rows['output_label_stem']):
            si=IMG_INDEX.get(img)
            if si is None: miss+=1; continue
            polys=read_poly(SEG_INDEX.get(stem))
            if DROP_BIG_LABELS and any(bbox_area(p[1])>=BIG_AREA_THR for p in polys):
                excl+=1; continue   # 오라벨 포함 이미지 통째 제외(무결성)
            n,d=slice_one(si,polys,split,stem)
            tot+=n; dtot+=d
        print(f'  {split}: {tot} tiles (결함타일 {dtot}, 오라벨이미지제외 {excl}, 원본누락 {miss})')
    done.write_text('ok')

import yaml
ct_tiled_yaml=TILED/'data.yaml'
ct_tiled_yaml.write_text(yaml.safe_dump({'path':str(TILED),'train':'images/train','val':'images/val','names':['porosity']},allow_unicode=True,sort_keys=False),encoding='utf-8')
print('tiled yaml:',ct_tiled_yaml,'| TILE=',TILE)

In [ ]:
# == §3 학습 (매 에폭 Drive 백업 + resume) ==
from ultralytics import YOLO
IMGSZ=TILE
BATCH=12   # OOM 나면 8로 내릴 것
local_run=Path('/content/runs_main')/RUN_NAME
CKPT=DRIVE_OUT/'runs_main'/RUN_NAME/'weights'; CKPT.mkdir(parents=True,exist_ok=True)

def _sync(trainer):   # 매 에폭 스냅샷/last/best를 Drive로 (세션 죽어도 회수)
    try:
        wdir=Path(trainer.last).parent
        for p in wdir.glob('epoch*.pt'):                     # 스냅샷은 불변 → 1회만 복사
            if not (CKPT/p.name).exists(): shutil.copy(p,CKPT/p.name)
        for p in (Path(trainer.last),Path(trainer.best)):
            if p.exists() and p.resolve()!=(CKPT/p.name).resolve(): shutil.copy(p,CKPT/p.name)
        if Path(trainer.best).exists(): shutil.copy(trainer.best,BACKUP)
    except Exception as e: print('동기화 스킵:',e)

# resume: Drive 체크포인트를 로컬로 미러. last.pt 없으면 최신 epoch 스냅샷을 승격
local_last=local_run/'weights'/'last.pt'
drun=DRIVE_OUT/'runs_main'/RUN_NAME
drive_ckpts=sorted(drun.rglob('*.pt'))
if not local_last.exists() and drive_ckpts:
    for p in drun.rglob('*'):
        if p.is_file():
            t=local_run/p.relative_to(drun); t.parent.mkdir(parents=True,exist_ok=True); shutil.copy(p,t)
    if not local_last.exists():
        eps=sorted((local_run/'weights').glob('epoch*.pt'), key=lambda p:int(''.join(filter(str.isdigit,p.stem)) or -1))
        assert eps, f'★ Drive에 .pt는 있는데 last/epoch 없음: {drun}'
        shutil.copy(eps[-1], local_last); print('last.pt 없음 → 최신 스냅샷 승격:', eps[-1].name)
    print('세션 재시작 감지 → Drive 체크포인트 로컬 복원:',local_last)

if local_last.exists():
    print('체크포인트 발견 → resume:',local_last)
    m=YOLO(str(local_last)); m.add_callback('on_fit_epoch_end',_sync); m.train(resume=True)
    run_dir=local_run
else:
    assert not drive_ckpts, f'★ Drive에 체크포인트 {len(drive_ckpts)}개 있는데 resume 못함 — 경로 확인(새로 시작 방지). 진짜 새로 시작하려면 이 assert 지우기'
    m=YOLO('yolo11m-seg.pt')   # plain yolo11m-seg (P2 헤드와의 A/B용 — 나머지 config 동일)
    m.add_callback('on_fit_epoch_end',_sync)
    # val=False: CT val mAP은 함정이라 안 씀. 판정은 §4 conf 스윕 F1.
    m.train(data=str(ct_tiled_yaml), imgsz=IMGSZ, batch=BATCH, cache='disk', val=False,
            epochs=CFG['epochs'], patience=CFG['patience'], save_period=SAVE_PERIOD,
            optimizer='AdamW', lr0=CFG['lr0'], cos_lr=True, amp=CFG['amp'],
            hsv_h=CFG['hsv_h'], hsv_s=CFG['hsv_s'], hsv_v=CFG['hsv_v'],
            degrees=CFG['degrees'], flipud=CFG['flipud'], translate=CFG['translate'],
            scale=CFG['scale'], copy_paste=CFG['copy_paste'],
            mask_ratio=CFG['mask_ratio'], overlap_mask=CFG['overlap_mask'],
            project=str(RUNS), name=RUN_NAME, exist_ok=True, seed=SEED)
    run_dir=RUNS/RUN_NAME

best=run_dir/'weights'/'best.pt'
final=best if best.exists() else (run_dir/'weights'/'last.pt' if (run_dir/'weights'/'last.pt').exists() else None)
assert final,'학습 산출물 없음'
dst=DRIVE_OUT/'runs_main'/RUN_NAME
if run_dir.resolve()!=dst.resolve():
    shutil.copytree(run_dir,dst,dirs_exist_ok=True)
shutil.copy(final,BACKUP)
print(f'[OK] best: {final} | Drive: {dst} | 백업 {BACKUP}')
print('▶ val 껐으므로 판정은 held-out 노트북으로: weights/epoch*.pt 각각 conf 스윕 → best 에폭 선택')

In [4]:
# == §4 동분포 eval: val 이미지에 SAHI conf 스윕 → 동분포 F1 ==
# 단독 실행 OK: §0·§1만 먼저(재학습 불필요).
!pip -q install sahi
import logging, warnings, gc, time, random
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
import torch

# sahi 로그 소음 차단(이미지당 수십 줄 찍음)
for _n in list(logging.root.manager.loggerDict):
    if _n.split('.')[0] in ('sahi', 'ultralytics'):
        _lg = logging.getLogger(_n); _lg.setLevel(logging.ERROR); _lg.propagate = False
logging.getLogger('sahi').setLevel(logging.ERROR)
logging.getLogger('sahi').propagate = False
warnings.filterwarnings('ignore')

# SAHI slice는 학습 TILE(1280)에 맞출 것 = train/test 해상도 일치
SLICE, OV, BASE_CONF, PP, IOU_HIT = 1280, 0.2, 0.02, 0.5, 0.1
TAG = f'slice{SLICE}_ov{OV}'   # 출력 파일명 구분용
SWEEP = [0.02, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40]
EVAL_EPOCHS = [6]   # 에폭 재선정하려면 [4,5,6,7]

# 평가 장수가 실행시간을 결정(이미지당 2~10초). 서브셋은 에폭 선택용, 헤드라인은 전량.
EVAL_N = None      # None = 전량. 숫자를 넣으면 서브셋
BALANCED = False   # 자연분포 유지 (balanced는 precision 낙관)
SEED = 42

# 가중치가 여러 폴더에 흩어져 있어 순서대로 탐색
MYDRIVE = Path('/content/drive/MyDrive')
WROOTS = [DRIVE_OUT/'runs_main'/RUN_NAME/'weights',
          MYDRIVE/'kt_out'/'runs_main'/RUN_NAME/'weights',
          ROOT/'runs_main'/RUN_NAME/'weights',
          PROJECT/'runs_main'/RUN_NAME/'weights']
for _d in sorted(MYDRIVE.glob('kt_out*')):
    _w = _d/'runs_main'/RUN_NAME/'weights'
    if _w not in WROOTS: WROOTS.append(_w)

print('■ 가중치 탐색 경로별 보유 에폭:')
for _r in WROOTS:
    if _r.exists():
        _eps = sorted(int(''.join(filter(str.isdigit, p.stem))) for p in _r.glob('epoch*.pt'))
        print(f'   ✅ {_r}\n      → epoch {_eps}')
    else:
        print(f'   ✗  {_r} (없음)')

# §6 챔피언 백업은 파일명이 달라 따로 탐색
CHDIRS = [MYDRIVE/'kt_out'/'champions', ROOT/'champions',
          DRIVE_OUT/'champions', MYDRIVE/'battery_yolo'/'champions']
print('■ 챔피언 백업 탐색:')
for _c in CHDIRS:
    _f = sorted(_c.glob('ct_samedist_CHAMPION_ep*.pt')) if _c.exists() else []
    print(('   OK  ' if _f else '   --  ') + str(_c) + (f'  -> {[p.name for p in _f]}' if _f else ''))

def find_epoch(e):
    nm = f'epoch{e}.pt'
    for root in WROOTS:
        if (root/nm).exists(): return root/nm
    for _c in CHDIRS:
        p = _c/f'ct_samedist_CHAMPION_ep{e}.pt'
        if p.exists(): return p
    for base in [MYDRIVE, PROJECT]:                        # 최후: 전체 탐색
        hits = [h for h in sorted(base.rglob(nm)) if RUN_NAME in str(h)]
        if hits: return hits[0]
    return None

def read_gt_norm(lp):
    b = []
    if lp is not None and lp.exists() and lp.stat().st_size:
        for ln in lp.read_text().splitlines():
            v = ln.split()
            if len(v) >= 7:
                xy = list(map(float, v[1:])); xs = xy[0::2]; ys = xy[1::2]
                b.append((min(xs), min(ys), max(xs), max(ys)))
    return b
def iou(a, b):
    ix0, iy0 = max(a[0], b[0]), max(a[1], b[1]); ix1, iy1 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, ix1-ix0) * max(0, iy1-iy0)
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter/ua if ua > 0 else 0.0

# val 목록 구성 + 서브샘플
ALL = []
for img, stem in zip(ct_split['val']['output_image_name'], ct_split['val']['output_label_stem']):
    ip = IMG_INDEX.get(img)
    if ip is None: continue
    lp = SEG_INDEX.get(stem); has = lp is not None and lp.stat().st_size > 0
    ALL.append((ip, lp, has))
assert ALL, 'val 이미지 0장'
pos = [x for x in ALL if x[2]]; neg = [x for x in ALL if not x[2]]
random.seed(SEED)
if EVAL_N is None:
    VITEMS = ALL
elif BALANCED:
    k = EVAL_N // 2
    VITEMS = random.sample(pos, min(k, len(pos))) + random.sample(neg, min(k, len(neg)))
    random.shuffle(VITEMS)
else:
    VITEMS = random.sample(ALL, min(EVAL_N, len(ALL)))
np_, nn_ = sum(x[2] for x in VITEMS), sum(not x[2] for x in VITEMS)
print(f'동분포 val 전체 {len(ALL)}장(양 {len(pos)}/음 {len(neg)}) → 평가 {len(VITEMS)}장 (양 {np_}/음 {nn_})')
if EVAL_N is not None and BALANCED:
    print('  ⚠️ balanced 서브셋은 precision이 낙관적으로 나옴(base-rate 효과, 0725 §6 확인).')
    print('     에폭 선택용으로만 쓰고, 최종 헤드라인은 peak 에폭에 EVAL_N=None으로 재실행할 것.')

def prf(tp, fp, fn):
    P = tp/(tp+fp) if tp+fp else 0.; R = tp/(tp+fn) if tp+fn else 0.
    return P, R, (2*P*R/(P+R) if P+R else 0.)

def sweep_weight(wp):
    m = AutoDetectionModel.from_pretrained(model_type='ultralytics', model_path=str(wp),
                                           confidence_threshold=BASE_CONF, device='cuda:0')
    RES = []; t0 = time.time()
    for i, (ip, lp, g) in enumerate(VITEMS):
        r = get_sliced_prediction(str(ip), m, slice_height=SLICE, slice_width=SLICE,
                                  overlap_height_ratio=OV, overlap_width_ratio=OV,
                                  postprocess_match_threshold=PP, verbose=0)
        W, H = r.image_width, r.image_height
        preds = [(o.bbox.to_xyxy(), o.score.value) for o in r.object_prediction_list]
        gts = [(x0*W, y0*H, x1*W, y1*H) for x0, y0, x1, y1 in read_gt_norm(lp)]
        RES.append((preds, gts, g))
        if (i+1) % 25 == 0 or i+1 == len(VITEMS):
            el = time.time()-t0; eta = el/(i+1)*(len(VITEMS)-i-1)
            print(f'    {i+1}/{len(VITEMS)}  경과 {el/60:.1f}분  남은 ~{eta/60:.1f}분', flush=True)
    del m; gc.collect(); torch.cuda.empty_cache()
    rows = []
    for conf in SWEEP:
        itp = ifp = ifn = ltp = lfn = lfp = 0
        for preds, gts, g in RES:
            pk = [(b, s) for b, s in preds if s >= conf]; fired = len(pk) > 0
            itp += fired and g; ifp += fired and not g; ifn += (not fired) and g
            if g:
                loc = any(iou(b, gb) > IOU_HIT for b, _ in pk for gb in gts); ltp += loc; lfn += not loc
            elif fired: lfp += 1
        iP, iR, iF = prf(itp, ifp, ifn); lF = prf(ltp, lfp, lfn)[2]
        rows.append(dict(conf=conf, iP=iP, iR=iR, iF=iF, lF=lF,
                         loc=(ltp/(ltp+lfn) if ltp+lfn else 0), ifp=ifp, ifn=ifn))
    return rows

best_overall = None; lines = []
for e in EVAL_EPOCHS:
    wp = find_epoch(e)
    if wp is None: print('스킵(없음): epoch', e); continue
    print(f'\n[epoch{e}] ← {wp}', flush=True)
    rows = sweep_weight(wp)
    print('='*72)
    print(f'[epoch{e}] 동분포 val {len(VITEMS)}장 @ slice{SLICE}/ov{OV}  (held-out과 동일 프로토콜)')
    print(f'{"conf":>5} | {"img_P":>6} {"img_R":>6} {"img_F1":>6} | {"loc_F1":>6} {"loc%":>6} | {"FP":>5} {"FN":>4}')
    for s in rows:
        print(f'{s["conf"]:>5.2f} | {s["iP"]:>6.3f} {s["iR"]:>6.3f} {s["iF"]:>6.3f} | '
              f'{s["lF"]:>6.3f} {s["loc"]:>6.1%} | {s["ifp"]:>5} {s["ifn"]:>4}')
    b = max(rows, key=lambda s: s['iF'])
    print(f'  → F1-max: conf {b["conf"]:.2f} P{b["iP"]:.3f}/R{b["iR"]:.3f}/F1 {b["iF"]:.3f} | '
          f'loc-F1 {b["lF"]:.3f}/loc {b["loc"]:.1%}')
    lines.append(f'epoch{e}: F1 {b["iF"]:.3f}@conf{b["conf"]:.2f} (P{b["iP"]:.3f}/R{b["iR"]:.3f}) loc-F1 {b["lF"]:.3f}')
    if best_overall is None or b['iF'] > best_overall[1]['iF']: best_overall = (f'epoch{e}', b)

print('\n' + '#'*72)
if best_overall:
    n, b = best_overall
    print(f'🏆 동분포 최고 = [{n}] conf {b["conf"]:.2f} → img F1 {b["iF"]:.3f} '
          f'(P{b["iP"]:.3f}/R{b["iR"]:.3f}) | loc-F1 {b["lF"]:.3f}')
    print(f'   평가셋 {len(VITEMS)}장' + (' (balanced 서브셋 = precision 낙관)' if EVAL_N and BALANCED else ' (전량)'))
    print(f'   ▶ 다음: EVAL_EPOCHS=[{n.replace("epoch","")}], EVAL_N=None 으로 재실행 = 헤드라인 확정')
    txt = '\n'.join(lines) + f'\n최고: {n} F1 {b["iF"]:.3f} (평가 {len(VITEMS)}장)'
    for _d in [DRIVE_OUT, ROOT, Path('/content')]:      # kt_out_1 읽기전용 대비
        try:
            _p = _d/f'ct_samedist_eval_{TAG}.txt'; _p.write_text(txt, encoding='utf-8')
            print('   요약 →', _p); break
        except Exception:
            continue
    else:
        print('   (요약 저장 실패 — 위 표를 복사해 두세요)')


■ 가중치 탐색 경로별 보유 에폭:
   ✅ /content/drive/MyDrive/battery_yolo/kt_out_1/runs_main/train_ct_tiled_v41_samedist/weights
      → epoch [0, 1, 2, 3]
   ✅ /content/drive/MyDrive/kt_out/runs_main/train_ct_tiled_v41_samedist/weights
      → epoch [4, 5, 6, 7]
   ✗  /content/drive/MyDrive/battery_yolo/runs_main/train_ct_tiled_v41_samedist/weights (없음)
   ✗  /content/drive/MyDrive/battery_yolo/빅프로젝트/runs_main/train_ct_tiled_v41_samedist/weights (없음)
■ 챔피언 백업 탐색:
   OK  /content/drive/MyDrive/kt_out/champions  -> ['ct_samedist_CHAMPION_ep6.pt']
   --  /content/drive/MyDrive/battery_yolo/champions
   --  /content/drive/MyDrive/battery_yolo/kt_out_1/champions
   --  /content/drive/MyDrive/battery_yolo/champions
동분포 val 전체 10140장(양 2083/음 8057) → 평가 10140장 (양 2083/음 8057)

[epoch6] ← /content/drive/MyDrive/kt_out/runs_main/train_ct_tiled_v41_samedist/weights/epoch6.pt
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/setti

In [6]:
# == §5 셀ID별 train/val 분포 확인 ==
# 전제: §0 실행(ct_split 존재). 모든 셀이 train·val 양쪽에 있어야 동분포.
import pandas as pd
tr = ct_split['train'].groupby('battery_id').size()
va = ct_split['val'].groupby('battery_id').size()
dist = pd.DataFrame({'train': tr, 'val': va}).fillna(0).astype(int)
dist.index = dist.index.astype(str)
dist = dist.loc[sorted(dist.index, key=lambda x: int(x) if x.isdigit() else 10**9)]
dist['total'] = dist['train'] + dist['val']
dist['val%']  = (dist['val'] / dist['total'] * 100).round(1)
print(dist.to_string())
print('-'*52)
both = ((dist['train'] > 0) & (dist['val'] > 0)).all()
print(f'셀 수: {len(dist)} (기대 47)')
print(f'val 0장(=1장뿐이라 train전용) 셀: {list(dist.index[dist["val"]==0]) or "없음"}')
print(f'train 0장 셀: {list(dist.index[dist["train"]==0]) or "없음"}')
print(f'✅ 모든 셀이 train·val 양쪽에 있나(동분포 핵심): {bool(both)}')
print(f'합계  train {int(dist["train"].sum())} / val {int(dist["val"].sum())}'
      f'  (val 비율 {dist["val"].sum()/dist["total"].sum():.1%})')

            train  val  total  val%
battery_id                         
101          1377  243   1620  15.0
102          1330  235   1565  15.0
103          1378  243   1621  15.0
104          1366  241   1607  15.0
105          1377  243   1620  15.0
106          1377  243   1620  15.0
107          1377  243   1620  15.0
108          1377  243   1620  15.0
109          1374  243   1617  15.0
110          1377  243   1620  15.0
111          1377  243   1620  15.0
112          1377  243   1620  15.0
113          1683  297   1980  15.0
114          1530  270   1800  15.0
115          1377  243   1620  15.0
116          1377  243   1620  15.0
117          1377  243   1620  15.0
118          1369  242   1611  15.0
119          1377  243   1620  15.0
120          1064  188   1252  15.0
121           768  136    904  15.0
122           764  135    899  15.0
123           741  131    872  15.0
124           768  135    903  15.0
125           687  121    808  15.0
126           706  124    83

In [ ]:
# == §6 챔피언 가중치 고정 백업 ==
# 전제: §0·§1 실행. 흩어진 가중치를 한 파일로 고정.
import shutil, hashlib, json as _json
from pathlib import Path

EPOCH = 6
TAG   = f'ct_samedist_CHAMPION_ep{EPOCH}'

# 1) 소스 찾기
MYDRIVE = Path('/content/drive/MyDrive')
ROOTS = [DRIVE_OUT/'runs_main'/RUN_NAME/'weights',
         MYDRIVE/'kt_out'/'runs_main'/RUN_NAME/'weights',
         ROOT/'runs_main'/RUN_NAME/'weights',
         PROJECT/'runs_main'/RUN_NAME/'weights']
SRC = next((r/f'epoch{EPOCH}.pt' for r in ROOTS if (r/f'epoch{EPOCH}.pt').exists()), None)
if SRC is None:
    hits = [h for h in sorted(MYDRIVE.rglob(f'epoch{EPOCH}.pt')) if RUN_NAME in str(h)]
    SRC = hits[0] if hits else None
assert SRC, f'★epoch{EPOCH}.pt 못찾음 — ROOTS 확인'
print('소스:', SRC, f'({SRC.stat().st_size/1e6:.1f} MB)')

# 2) 쓰기 가능한 목적지(읽기전용 폴더 대비)
def writable(*cands):
    for c in cands:
        try:
            c.mkdir(parents=True, exist_ok=True)
            t = c/'.wtest'; t.write_text('ok'); t.unlink(); return c
        except Exception: continue
    return None
DST_DIR = writable(MYDRIVE/'kt_out'/'champions', ROOT/'champions',
                   DRIVE_OUT/'champions', MYDRIVE/'kt_out_mine'/'champions')
assert DST_DIR, '★쓰기 가능한 백업 폴더 없음'
DST = DST_DIR/f'{TAG}.pt'

# 3) 복사 + 해시 검증(조용히 깨진 복사 방지)
def sha(p, buf=1 << 20):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        while (b := f.read(buf)): h.update(b)
    return h.hexdigest()

if not DST.exists() or DST.stat().st_size != SRC.stat().st_size:
    shutil.copy2(SRC, DST)
s_src, s_dst = sha(SRC), sha(DST)
assert s_src == s_dst, f'★복사본 해시 불일치! {s_src[:12]} vs {s_dst[:12]} — 다시 실행'
print(f'✅ 백업: {DST}  sha256 {s_dst[:16]}…  검증 통과')

# 4) 메타 사이드카(운영점 숫자를 가중치 옆에 보관)
meta = {
    'champion': f'samedist plain yolo11m-seg epoch{EPOCH}',
    'run': RUN_NAME,
    'split': '동분포(known-type) 이미지단위 stratified — 47셀 전부 train·val 양쪽, 셀당 15% val, seed42',
    'data_policy': 'v4.1 nobig (대형 ≥25% 라벨 이미지 제외)',
    'inference': 'SAHI sliced slice=1024 / overlap=0.4',
    'superseded_by': {
        'inference': 'SAHI sliced slice=1280 / overlap=0.2',
        'reason': '학습 TILE(1280)과 정합. 1024/0.4는 recall↓·오탐 2배(0728)',
        'deploy_conf': 0.05,
        'note': ('아래 operating_points 는 전부 1024/0.4 에서 잰 값이라 배포 운영점이 아니다. '
                 '배포 계약은 slice1280/ov0.2 @ conf 0.05 이며 ct_output_schema.sample.json 이 정본.'),
    },
    'eval_set': 'samedist val 전량 10,140장 (양 2,083 / 음 8,057)',
    'operating_points': {
        'report_f1max': {'conf': 0.15, 'P': 0.855, 'R': 0.905, 'F1': 0.879,
                         'loc_F1': 0.842, 'loc_pct': 0.839, 'FP': 320, 'FN': 197},
        'gate_high_recall': {'conf': 0.05, 'P': 0.722, 'R': 0.969, 'F1': 0.827, 'FP': 777},
    },
    'balanced_400_reference': {'conf': 0.05, 'F1': 0.925, 'note': 'precision 낙관 — 헤드라인 아님'},
    'runner_up': {'epoch': 7, 'F1': 0.862, 'loc_F1': 0.815, 'note': '고recall형(@0.02 R0.994), F1·loc 모두 열세'},
    'fp_rate_on_normal': round(320/8057, 4),
    'caveat': ('peak 에폭을 이 val로 선택 → 약한 낙관(plateau 평탄해 위험 작음). '
               '최적 conf는 배포 분포에 따라 이동(balanced 0.05 vs 자연분포 0.15).'),
    'headline': '알려진 타입 재검사 F1 0.879 (R 0.905, loc 84%) / 완전 새 타입 0.781로 재보정 필요',
    'src': str(SRC), 'sha256': s_dst,
}
DST.with_suffix('.json').write_text(_json.dumps(meta, ensure_ascii=False, indent=2), encoding='utf-8')
print('✅ 메타:', DST.with_suffix('.json'))
print(_json.dumps(meta['operating_points'], ensure_ascii=False, indent=2))
print(f"\n▶ 앞으로 이 파일만 쓰면 됨: {DST}")


In [7]:
# == §7 A/B 실험용 예측 캐시 (GPU 1회) ==
# 전제: §0·§1만.
# 저임계(conf 0.02) 예측을 디스크에 덤프 → §8을 GPU 없이 몇 번이든 재실행.
!pip -q install sahi
import re, json as _json, logging, warnings, gc, time, random
from pathlib import Path
import numpy as np
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
import torch

for _n in list(logging.root.manager.loggerDict):
    if _n.split('.')[0] in ('sahi', 'ultralytics'):
        _lg = logging.getLogger(_n); _lg.setLevel(logging.ERROR); _lg.propagate = False
warnings.filterwarnings('ignore')

SLICE, OV, BASE_CONF, PP = 1024, 0.4, 0.02, 0.5   # 이 A/B 캐시는 구 프로토콜. 배포·§4는 1280/0.2
N_POS, N_NEG, K_NBR, SEED = 200, 400, 1, 42   # 중심 장수 / 이웃 반경. 추론량 = (N_POS+N_NEG)*(1+2*K_NBR)
EPOCH = 6

# 1) 슬라이스 인덱스 파싱 + stride 진단
STEM_COL = next((c for c in ['original_stem','output_label_stem','output_image_name'] if c in dev.columns), None)
assert STEM_COL, 'stem 컬럼 없음'
d = dev.copy()
d[['_ax','_idx']] = d[STEM_COL].str.replace(r'\.[a-zA-Z]+$','',regex=True).str.extract(r'_([xyz])_(\d+)$')
d = d[d['_idx'].notna()].copy(); d['_idx'] = d['_idx'].astype(int)
assert len(d) > 0.9*len(dev), f'슬라이스 인덱스 파싱 실패 {len(d)}/{len(dev)} — {STEM_COL} 형식 확인'

_st = [np.median(np.diff(np.sort(g['_idx'].unique())))
       for _, g in d.groupby(['battery_id','_ax']) if g['_idx'].nunique() > 1]
STRIDE = float(np.median(_st)); STEP = max(1, int(round(STRIDE)))
print(f'■ 슬라이스 stride 중앙값 = {STRIDE:.1f} (축·셀 {len(_st)}조합)')
if STRIDE <= 2:
    print('   ✅ 인접 슬라이스 존재 → A(3D 연속성) 유효')
else:
    print(f'   ⚠️ 성김(stride {STEP}) → A는 "인접"이 아니라 {STEP}장 건너뛴 이웃. 효과 약할 수 있음.')
    print('      A가 안 들으면 B만 채택하고 넘어갈 것.')

# 2) 중심 = val 이미지, 이웃 = dev 전체(train 포함)
# 이웃이 train 이미지면 3D 지지가 낙관 편향(§8이 train 비율을 찍어줌).
# itertuples는 밑줄로 시작하는 컬럼(_ax)을 _1,_2로 바꿔버림 → zip으로 직접
NBR = {(c, a, int(i)): n for c, a, i, n in
       zip(d['battery_id'], d['_ax'], d['_idx'], d['output_image_name'])}
TRAIN_NAMES = set(ct_split['train']['output_image_name'])
vkey = set(ct_split['val']['output_image_name'])
cands = [(k, n) for k, n in NBR.items() if n in vkey]
pos = [(k,n) for k,n in cands if (SEG_INDEX.get(n.rsplit('.',1)[0]) or Path('/nope')).exists()
       and SEG_INDEX[n.rsplit('.',1)[0]].stat().st_size > 0]
_pn = {n for _,n in pos}
neg = [(k,n) for k,n in cands if n not in _pn]
random.seed(SEED)
CENTERS = random.sample(pos, min(N_POS,len(pos))) + random.sample(neg, min(N_NEG,len(neg)))
print(f'중심: 양 {min(N_POS,len(pos))} / 음 {min(N_NEG,len(neg))} (val 양 {len(pos)}/음 {len(neg)})')

WANT = {}                                   # name -> (cid, ax, idx, is_center)
for (cid,ax,ix), nm in CENTERS:
    WANT[nm] = (cid,ax,ix,True)
    for k in range(1, K_NBR+1):
        for j in (ix-STEP*k, ix+STEP*k):
            n2 = NBR.get((cid,ax,j))
            if n2 and n2 not in WANT: WANT[n2] = (cid,ax,j,False)
ITEMS = [(nm,v) for nm,v in WANT.items() if nm in IMG_INDEX]
nc = sum(v[3] for _,v in ITEMS)
print(f'추론 대상 {len(ITEMS)}장 (중심 {nc} + 이웃 {len(ITEMS)-nc}) | 이웃 중 train 출신 '
      f'{sum(1 for n,v in ITEMS if not v[3] and n in TRAIN_NAMES)}장')
print(f'▶ 예상 시간: 장당 2초면 ~{len(ITEMS)*2/60:.0f}분, 5초면 ~{len(ITEMS)*5/60:.0f}분 '
      f'(25장마다 실측 ETA 출력 — 너무 길면 중단하고 N_NEG↓)')

# 3) 챔피언 가중치
MYDRIVE = Path('/content/drive/MyDrive')
_c = [MYDRIVE/'kt_out'/'champions'/f'ct_samedist_CHAMPION_ep{EPOCH}.pt',
      ROOT/'champions'/f'ct_samedist_CHAMPION_ep{EPOCH}.pt',
      DRIVE_OUT/'champions'/f'ct_samedist_CHAMPION_ep{EPOCH}.pt']
_c += [r/f'epoch{EPOCH}.pt' for r in
       [DRIVE_OUT/'runs_main'/RUN_NAME/'weights', MYDRIVE/'kt_out'/'runs_main'/RUN_NAME/'weights',
        ROOT/'runs_main'/RUN_NAME/'weights', PROJECT/'runs_main'/RUN_NAME/'weights']]
WP = next((p for p in _c if p.exists()), None)
if WP is None:
    _h = [h for h in sorted(MYDRIVE.rglob(f'epoch{EPOCH}.pt')) if RUN_NAME in str(h)]; WP = _h[0] if _h else None
assert WP, f'★epoch{EPOCH} 가중치 못찾음'
print('가중치:', WP)

# 4) 추론 → 캐시
def _gt(nm):
    lp = SEG_INDEX.get(nm.rsplit('.',1)[0])
    out = []
    if lp is not None and lp.exists() and lp.stat().st_size:
        for ln in lp.read_text().splitlines():
            v = ln.split()
            if len(v) >= 7:
                xy = list(map(float, v[1:])); xs, ys = xy[0::2], xy[1::2]
                out.append([min(xs),min(ys),max(xs),max(ys)])
    return out

m = AutoDetectionModel.from_pretrained(model_type='ultralytics', model_path=str(WP),
                                       confidence_threshold=BASE_CONF, device='cuda:0')
CACHE = {}; t0 = time.time()
for i,(nm,(cid,ax,ix,isc)) in enumerate(ITEMS):
    r = get_sliced_prediction(str(IMG_INDEX[nm]), m, slice_height=SLICE, slice_width=SLICE,
                              overlap_height_ratio=OV, overlap_width_ratio=OV,
                              postprocess_match_threshold=PP, verbose=0)
    W,H = r.image_width, r.image_height
    g = _gt(nm)
    CACHE[nm] = dict(cid=cid, ax=ax, idx=ix, center=bool(isc), W=W, H=H,
                     train=nm in TRAIN_NAMES,
                     preds=[[*map(float,o.bbox.to_xyxy()), float(o.score.value)] for o in r.object_prediction_list],
                     gts=[[x0*W,y0*H,x1*W,y1*H] for x0,y0,x1,y1 in g], has=bool(g))
    if (i+1) % 25 == 0 or i+1 == len(ITEMS):
        el = time.time()-t0
        print(f'  {i+1}/{len(ITEMS)}  경과 {el/60:.1f}분  남은 ~{el/(i+1)*(len(ITEMS)-i-1)/60:.1f}분', flush=True)
del m; gc.collect(); torch.cuda.empty_cache()

def _w(*cs):
    for c in cs:
        try:
            c.mkdir(parents=True, exist_ok=True); t=c/'.wt'; t.write_text('ok'); t.unlink(); return c
        except Exception: continue
CDIR = _w(MYDRIVE/'kt_out'/'ab_cache', ROOT/'ab_cache', DRIVE_OUT/'ab_cache', Path('/content/ab_cache'))
CACHE_PATH = CDIR/f'ct_samedist_ep{EPOCH}_preds.json'
CACHE_PATH.write_text(_json.dumps({'meta': dict(epoch=EPOCH, weight=str(WP), slice=SLICE, ov=OV,
                                                base_conf=BASE_CONF, pp=PP, stride=STRIDE, step=STEP,
                                                k_nbr=K_NBR, n_val_pos=len(pos), n_val_neg=len(neg)),
                                   'cache': CACHE}), encoding='utf-8')
print(f'\n✅ 캐시 {len(CACHE)}장 → {CACHE_PATH}  ({CACHE_PATH.stat().st_size/1e6:.1f} MB)')
print('▶ 다음: §8(A·B 분석) — GPU 불필요, 몇 초. 파라미터 바꿔 몇 번이든 재실행 가능.')


■ 슬라이스 stride 중앙값 = 1.0 (축·셀 134조합)
   ✅ 인접 슬라이스 존재 → A(3D 연속성) 유효
중심: 양 200 / 음 400 (val 양 2083/음 8057)
추론 대상 1661장 (중심 600 + 이웃 1061) | 이웃 중 train 출신 898장
▶ 예상 시간: 장당 2초면 ~55분, 5초면 ~138분 (25장마다 실측 ETA 출력 — 너무 길면 중단하고 N_NEG↓)
가중치: /content/drive/MyDrive/kt_out/champions/ct_samedist_CHAMPION_ep6.pt
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
  25/1661  경과 0.3분  남은 ~19.6분
  50/1661  경과 0.5분  남은 ~15.6분
  75/1661  경과 0.6분  남은 ~13.1분
  100/1661  경과 0.8분  남은 ~12.1분
  125/1661  경과 1.0분  남은 ~11.7분
  150/1661  경과 1.1분  남은 ~11.4분
  175/1661  경과 1.3분  남은 ~11.0분
  200/1661  경과 1.5분  남은 ~10.6분
  225/1661  경과 1.6분  남은 ~10.5분
  250/1661  경과 1.8분  남은 ~10.0분
  275/1661  경과 1.9분  남은 ~9.7분
  300/1661  경과 2.1분  남은 ~9.5분
  325/1661  경과 2.3분  남

In [8]:
# == §8 A(3D 연속성) + B(중복병합 × min_dets) 분석 — CPU only ==
# 전제: §0 + §7(캐시 파일).
import json as _json, itertools
from pathlib import Path
import numpy as np

MYDRIVE = Path('/content/drive/MyDrive'); EPOCH = 6
_p = [MYDRIVE/'kt_out'/'ab_cache', ROOT/'ab_cache', DRIVE_OUT/'ab_cache', Path('/content/ab_cache')]
CP = next((c/f'ct_samedist_ep{EPOCH}_preds.json' for c in _p if (c/f'ct_samedist_ep{EPOCH}_preds.json').exists()), None)
assert CP, '★캐시 없음 — §7 먼저'
_o = _json.loads(CP.read_text(encoding='utf-8')); META, CACHE = _o['meta'], _o['cache']
STEP, K_NBR = META['step'], META['k_nbr']
print(f'캐시 {len(CACHE)}장 | stride {META["stride"]:.1f} (STEP {STEP}, K {K_NBR}) | {CP.name}')

SWEEP     = [0.02, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40]
MIN_DETS  = [1, 2, 3]
MERGE_IOU = 0.2      # 이보다 겹치면 같은 결함으로 병합 (SAHI 이음매 중복 대응)
NBR_IOU   = 0.1      # 이웃 슬라이스에서 이만큼 겹치면 3D 지지로 인정
IOU_HIT   = 0.1      # loc 판정
PI_POS, PI_NEG = 2083, 8057   # 동분포 val 전량의 양/음 (자연분포 환산용)

def iou(a,b):
    ix0,iy0 = max(a[0],b[0]), max(a[1],b[1]); ix1,iy1 = min(a[2],b[2]), min(a[3],b[3])
    it = max(0,ix1-ix0)*max(0,iy1-iy0)
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - it
    return it/ua if ua > 0 else 0.

def merge(dets, thr):
    """겹치는 박스를 union 박스 1개로(점수=max). min_dets를 쓰려면 필수 — 안 하면
       SAHI 이음매 중복이 '검출 2개'로 세어져 min_dets가 무력화됨."""
    n = len(dets)
    if n < 2: return [list(d) for d in dets]
    par = list(range(n))
    def f(x):
        while par[x] != x: par[x] = par[par[x]]; x = par[x]
        return x
    for i,j in itertools.combinations(range(n),2):
        if iou(dets[i][:4], dets[j][:4]) > thr: par[f(i)] = f(j)
    grp = {}
    for i,dt in enumerate(dets): grp.setdefault(f(i), []).append(dt)
    return [[min(g_[0] for g_ in G), min(g_[1] for g_ in G), max(g_[2] for g_ in G),
             max(g_[3] for g_ in G), max(g_[4] for g_ in G)] for G in grp.values()]

KEY = {(v['cid'], v['ax'], v['idx']): k for k, v in CACHE.items()}
def supported(rec, box, conf):
    """이웃 슬라이스(±STEP..±STEP*K)에 겹치는 검출이 있으면 3D 지지.
       이웃이 캐시에 없으면(경계 슬라이스 등) fail-open = 지지된 것으로 간주."""
    seen = False
    for k in range(1, K_NBR+1):
        for j in (rec['idx']-STEP*k, rec['idx']+STEP*k):
            nk = KEY.get((rec['cid'], rec['ax'], j))
            if nk is None: continue
            seen = True
            if any(d[4] >= conf and iou(box, d[:4]) > NBR_IOU for d in CACHE[nk]['preds']):
                return True
    return not seen

CENTERS = [v for v in CACHE.values() if v['center']]
NP_, NN_ = sum(v['has'] for v in CENTERS), sum(not v['has'] for v in CENTERS)
_nb = [v for v in CACHE.values() if not v['center']]
print(f'중심 {len(CENTERS)}장 (양 {NP_}/음 {NN_}) | 이웃 {len(_nb)}장 중 train 출신 '
      f'{sum(v["train"] for v in _nb)}장 → 3D 지지가 낙관될 수 있는 비율 '
      f'{(sum(v["train"] for v in _nb)/max(1,len(_nb))):.0%}')

def run(mode, conf, md):
    """mode: base(원본) / merge(병합) / merge3d(병합+3D연속성). 반환: 이미지별 fired + loc."""
    out = []
    for v in CENTERS:
        dts = [d for d in v['preds'] if d[4] >= conf]
        if mode != 'base': dts = merge(dts, MERGE_IOU)
        if mode == 'merge3d': dts = [d for d in dts if supported(v, d[:4], conf)]
        fired = len(dts) >= md
        loc = fired and v['has'] and any(iou(d[:4], g) > IOU_HIT for d in dts for g in v['gts'])
        out.append((v['has'], fired, loc))
    return out

def stats(res):
    tp = sum(h and f for h,f,_ in res); fn = sum(h and not f for h,f,_ in res)
    fp = sum((not h) and f for h,f,_ in res)
    R  = tp/max(1,tp+fn); FPR = fp/max(1,NN_)
    # 자연분포로 환산: R·FPR은 분포 불변이라 개수만 다시 곱하면 됨
    T, F, N = R*PI_POS, FPR*PI_NEG, (1-R)*PI_POS
    P  = T/max(1e-9, T+F); F1 = 2*T/max(1e-9, 2*T+F+N)
    lt = sum(l for _,_,l in res); lfn = sum(h and not l for h,_,l in res)
    lF = 2*lt/max(1e-9, 2*lt+fp+lfn)
    return dict(R=R, FPR=FPR, P=P, F1=F1, locF=lF, loc=lt/max(1,NP_), tp=tp, fp=fp, fn=fn)

BASE = stats(run('base', 0.15, 1))
print(f'\n[기준] base conf0.15 md1 → R {BASE["R"]:.3f} FPR {BASE["FPR"]:.2%} '
      f'| 자연분포환산 P {BASE["P"]:.3f} F1 {BASE["F1"]:.3f} (전량실측 P0.855/F1 0.879와 대조)')

ROWS = []
for mode in ['base','merge','merge3d']:
    print(f'\n{"="*88}\n■ {mode}   (자연분포 환산: 양 {PI_POS}/음 {PI_NEG})')
    print(f'{"conf":>5} {"md":>3} | {"R":>6} {"FPR":>7} | {"P":>6} {"F1":>6} {"locF1":>6} | {"ΔF1":>6}')
    for md in MIN_DETS:
        for conf in SWEEP:
            s = stats(run(mode, conf, md)); s.update(mode=mode, conf=conf, md=md); ROWS.append(s)
            print(f'{conf:>5.2f} {md:>3} | {s["R"]:>6.3f} {s["FPR"]:>7.2%} | {s["P"]:>6.3f} '
                  f'{s["F1"]:>6.3f} {s["locF"]:>6.3f} | {s["F1"]-BASE["F1"]:>+6.3f}')

B = max(ROWS, key=lambda s: s['F1'])
print(f'\n{"#"*88}')
print(f'🏆 최고: {B["mode"]} conf{B["conf"]:.2f} min_dets={B["md"]} → F1 {B["F1"]:.3f} '
      f'(base 0.15/md1 대비 {B["F1"]-BASE["F1"]:+.3f}) | R {B["R"]:.3f} FPR {B["FPR"]:.2%} locF1 {B["locF"]:.3f}')

# 페어드 비교: 같은 이미지에서 뭐가 죽고 살았나(표본 작아도 신뢰 가능)
r0 = run('base', 0.15, 1); r1 = run(B['mode'], B['conf'], B['md'])
fp_kill = sum((not h) and f0 and not f1 for (h,f0,_),(_,f1,_) in zip(r0,r1))
fp_new  = sum((not h) and (not f0) and f1 for (h,f0,_),(_,f1,_) in zip(r0,r1))
tp_lost = sum(h and f0 and not f1 for (h,f0,_),(_,f1,_) in zip(r0,r1))
tp_gain = sum(h and (not f0) and f1 for (h,f0,_),(_,f1,_) in zip(r0,r1))
print(f'\n페어드(같은 이미지 기준): FP 제거 {fp_kill} / FP 추가 {fp_new} | TP 손실 {tp_lost} / TP 회복 {tp_gain}')
print(f'  교환비 = FP제거 {fp_kill} : TP손실 {tp_lost}'
      + (f' = {fp_kill/tp_lost:.1f}' if tp_lost else ' (TP 손실 0 = 공짜 이득)'))
print(f'  손익분기 = (양성수+FP)/TP = {(PI_POS+BASE["FPR"]*PI_NEG)/max(1e-9,BASE["R"]*PI_POS):.2f} '
      '→ 교환비가 이보다 크면 자연분포 F1 상승')
if fp_kill + tp_lost < 10:
    print('  ⚠️ 변화 건수 10 미만 = 표본 부족. N_NEG 올려 §7 재실행 후 재판정할 것.')

# 실제 라인 불량률 2% 가정 시
print(f'\n[불량률 2% 라인 환산] 1만장당')
for tag, s in [('현행 base 0.15/md1', BASE), (f'{B["mode"]} {B["conf"]:.2f}/md{B["md"]}', B)]:
    T = s['R']*200; F = s['FPR']*9800
    print(f'  {tag:<28} 검출 {T:>5.0f}/200  오탐 {F:>5.0f}  precision {T/max(1e-9,T+F):.3f}  재검부하 {(T+F)/100:.1f}%')

for _d in [MYDRIVE/'kt_out', ROOT, Path('/content')]:
    try:
        _f = _d/'ct_ab_result.txt'
        _f.write_text('\n'.join(f'{r["mode"]:>8} conf{r["conf"]:.2f} md{r["md"]} R{r["R"]:.3f} '
                                f'FPR{r["FPR"]:.4f} F1{r["F1"]:.3f} locF1{r["locF"]:.3f}' for r in ROWS),
                      encoding='utf-8')
        print('\n요약 →', _f); break
    except Exception: continue


캐시 1661장 | stride 1.0 (STEP 1, K 1) | ct_samedist_ep6_preds.json
중심 600장 (양 200/음 400) | 이웃 1061장 중 train 출신 898장 → 3D 지지가 낙관될 수 있는 비율 85%

[기준] base conf0.15 md1 → R 0.830 FPR 5.75% | 자연분포환산 P 0.789 F1 0.809 (전량실측 P0.855/F1 0.879와 대조)

■ base   (자연분포 환산: 양 2083/음 8057)
 conf  md |      R     FPR |      P     F1  locF1 |    ΔF1
 0.02   1 |  0.975  20.25% |  0.555  0.707  0.765 | -0.102
 0.05   1 |  0.950  12.25% |  0.667  0.784  0.811 | -0.025
 0.10   1 |  0.900   7.50% |  0.756  0.822  0.824 | +0.013
 0.15   1 |  0.830   5.75% |  0.789  0.809  0.807 | +0.000
 0.20   1 |  0.770   4.50% |  0.816  0.792  0.782 | -0.017
 0.25   1 |  0.710   3.75% |  0.830  0.765  0.746 | -0.043
 0.30   1 |  0.640   3.25% |  0.836  0.725  0.713 | -0.084
 0.40   1 |  0.410   1.75% |  0.858  0.555  0.557 | -0.254
 0.02   2 |  0.825  10.00% |  0.681  0.746  0.776 | -0.063
 0.05   2 |  0.715   4.25% |  0.813  0.761  0.771 | -0.048
 0.10   2 |  0.610   2.50% |  0.863  0.715  0.712 | -0.094
 0.15   2 |  0.540   

In [9]:
v = ct_split['val']
same = sum(1 for a,b in zip(v['output_image_name'], v['output_label_stem'])
           if a.rsplit('.',1)[0] == b)
print(f'키 규약 일치: {same}/{len(v)}')
def npos(keys): return sum(1 for k in keys if SEG_INDEX.get(k) and SEG_INDEX[k].stat().st_size > 0)
print(f'label_stem 기준 양성 {npos(v["output_label_stem"])} | '
      f'image_stem 기준 양성 {npos([n.rsplit(".",1)[0] for n in v["output_image_name"]])} | 전량실측 2083')
print(f'dev 행 {len(d)} vs 고유 (셀,축,idx) {len(NBR)} → 키 충돌 {len(d)-len(NBR)}')
print(f'§7 가중치 확인: {WP}')

키 규약 일치: 10140/10140
label_stem 기준 양성 2083 | image_stem 기준 양성 2083 | 전량실측 2083
dev 행 67605 vs 고유 (셀,축,idx) 67605 → 키 충돌 0
§7 가중치 확인: /content/drive/MyDrive/kt_out/champions/ct_samedist_CHAMPION_ep6.pt


In [5]:
d = dev.copy()
d[['_ax','_idx']] = d['original_stem'].str.extract(r'_([xyz])_(\d+)$')
key = list(zip(d['battery_id'], d['_ax'], d['_idx']))
print(f'dev 행 {len(d)} vs 고유 (셀,축,슬라이스) {len(set(key))} → 중복 {len(d)-len(set(key))}')

dev 행 67605 vs 고유 (셀,축,슬라이스) 67605 → 중복 0
